# Complete PySpark Tutorial

PySpark is the Python API for Apache Spark, a unified analytics engine for large-scale data processing.

**Key features:**
- **In-memory computation** – up to 100x faster than Hadoop MapReduce
- **Distributed processing** – scales from one laptop to thousands of servers
- **Unified API** – SQL, DataFrames, streaming, machine learning, graph processing
- **Fault tolerance** – automatic recovery from failures
- **Multiple languages** – Python, Scala, Java, R, SQL

This tutorial covers everything from basic RDDs to advanced Structured Streaming and MLlib.

## Installation

```bash
pip install pyspark
```

Or with specific version:
```bash
pip install pyspark==3.5.0
```

For Jupyter integration:
```bash
pip install pyspark[notebook]
```

Note: PySpark requires Java 8 or 11. Download from [OpenJDK](https://adoptium.net/).

In [ ]:
# Initialize SparkSession (entry point for Spark)
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Create Spark session
spark = SparkSession.builder \
    .appName("CompletePySparkTutorial") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Set log level to WARN to reduce verbosity
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")

## 1. RDD (Resilient Distributed Dataset) – Low‑Level API

RDDs are the fundamental building blocks. They are distributed, fault-tolerant collections of objects.

In [ ]:
# Creating RDDs
sc = spark.sparkContext

# From parallelized collection
rdd = sc.parallelize([1, 2, 3, 4, 5])
print(f"RDD count: {rdd.count()}")
print(f"RDD collect: {rdd.collect()}")

# From external file
lines_rdd = sc.textFile("path/to/file.txt")  # HDFS, S3, local file

# Transformations
squared = rdd.map(lambda x: x ** 2)
print(f"Squared: {squared.collect()}")

filtered = rdd.filter(lambda x: x % 2 == 0)
print(f"Even numbers: {filtered.collect()}")

# Reduce
sum_rdd = rdd.reduce(lambda a, b: a + b)
print(f"Sum: {sum_rdd}")

# Key-value operations
pairs = sc.parallelize([("a", 1), ("b", 2), ("a", 3), ("c", 4)])
counts = pairs.reduceByKey(lambda a, b: a + b)
print(f"Reduce by key: {counts.collect()}")

## 2. DataFrames – High‑Level API (Recommended)

DataFrames provide optimization (Catalyst optimizer, Tungsten execution) and a SQL-like interface.

In [ ]:
# From list of tuples
data = [("Alice", 25, "Engineering"),
        ("Bob", 30, "Sales"),
        ("Charlie", 35, "Engineering"),
        ("Diana", 28, "Marketing"),
        ("Eve", 40, "Sales")]

columns = ["Name", "Age", "Department"]
df = spark.createDataFrame(data, columns)
df.show()

In [ ]:
# Basic DataFrame operations
print("Schema:")
df.printSchema()

print("\nCount:", df.count())

print("\nColumns:", df.columns)

print("\nFirst row:")
df.show(1)

print("\nHead (2 rows):")
df.head(2)

In [ ]:
# Selection and filtering
df.select("Name", "Age").show()

df.filter(df.Age > 30).show()

# Multiple conditions
df.filter((df.Age > 25) & (df.Department == "Engineering")).show()

# Using SQL expression
df.where("Age > 30 AND Department = 'Sales'").show()

In [ ]:
# Aggregations
df.groupBy("Department").agg(
    F.avg("Age").alias("avg_age"),
    F.count("*").alias("count"),
    F.max("Age").alias("max_age")
).show()

# Built-in aggregation functions
df.agg(
    F.mean("Age").alias("mean_age"),
    F.min("Age").alias("min_age"),
    F.max("Age").alias("max_age"),
    F.stddev("Age").alias("stddev_age")
).show()

In [ ]:
# Sorting
df.orderBy("Age", ascending=False).show()

df.sort("Department", "Age").show()

## 3. Spark SQL

Register DataFrames as temporary views and run SQL queries.

In [ ]:
# Register as temporary view
df.createOrReplaceTempView("employees")

# Run SQL query
result = spark.sql("""
    SELECT Department, COUNT(*) as count, AVG(Age) as avg_age
    FROM employees
    WHERE Age > 25
    GROUP BY Department
    ORDER BY avg_age DESC
""")
result.show()

## 4. Advanced Column Operations

Creating new columns, conditional logic, string operations, arrays, and maps.

In [ ]:
# Create new columns
df.withColumn("AgeGroup", 
    F.when(df.Age < 30, "Young")
     .when(df.Age < 40, "Middle")
     .otherwise("Senior")
).show()

# String operations
df.withColumn("NameUpper", F.upper(df.Name)) \
  .withColumn("NameLength", F.length(df.Name)) \
  .show()

# Working with arrays and maps
array_df = spark.createDataFrame([([1, 2, 3],), ([4, 5],)], ["nums"])
array_df.withColumn("first", F.col("nums")[0]) \
         .withColumn("size", F.size("nums")) \
         .show()

## 5. Window Functions

Window functions allow you to perform calculations across rows (e.g., running totals, rankings, lag/lead).

In [ ]:
# Sample sales data
sales_data = [("2024-01-01", "Product A", 100),
              ("2024-01-02", "Product A", 150),
              ("2024-01-03", "Product A", 200),
              ("2024-01-01", "Product B", 80),
              ("2024-01-02", "Product B", 120),
              ("2024-01-03", "Product B", 90)]
sales_df = spark.createDataFrame(sales_data, ["Date", "Product", "Sales"])

# Define window specification
window_spec = Window.partitionBy("Product").orderBy("Date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Running total
sales_df.withColumn("RunningTotal", F.sum("Sales").over(window_spec)).show()

# Ranking
sales_df.withColumn("Rank", F.rank().over(Window.partitionBy("Product").orderBy(F.desc("Sales")))).show()

# Lag and Lead
sales_df.withColumn("PrevDaySales", F.lag("Sales", 1).over(Window.partitionBy("Product").orderBy("Date"))).show()

## 6. Joins

PySpark supports all standard SQL join types: inner, left, right, outer, semi, anti, cross.

In [ ]:
df1 = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
df2 = spark.createDataFrame([(1, "Engineering"), (3, "Sales")], ["id", "dept"])

print("Inner join:")
df1.join(df2, on="id", how="inner").show()

print("Left join:")
df1.join(df2, on="id", how="left").show()

print("Right join:")
df1.join(df2, on="id", how="right").show()

print("Full outer join:")
df1.join(df2, on="id", how="outer").show()

print("Cross join (Cartesian product):")
df1.crossJoin(df2).show()

## 7. Handling Missing Data

Drop rows with nulls, fill with default values, or use imputation.

In [ ]:
missing_data = [("Alice", 25, None),
                ("Bob", None, "Sales"),
                (None, 30, "Engineering")]
df_missing = spark.createDataFrame(missing_data, ["Name", "Age", "Department"])
df_missing.show()

# Drop rows with any null
df_missing.dropna().show()

# Drop rows with null in specific columns
df_missing.dropna(subset=["Age"]).show()

# Fill nulls
df_missing.fillna({"Name": "Unknown", "Age": 0, "Department": "Unknown"}).show()

# Fill with column mean (requires computing mean first)
mean_age = df_missing.select(F.mean("Age")).collect()[0][0]
df_missing.fillna(mean_age, subset=["Age"]).show()

## 8. Reading and Writing Data

PySpark can read/write many formats: CSV, JSON, Parquet, Avro, ORC, JDBC, Hive, etc.

In [ ]:
# Write sample DataFrame to Parquet (recommended for performance)
df.write.parquet("employees.parquet", mode="overwrite")
print("Written to employees.parquet")

# Read from Parquet
df_parquet = spark.read.parquet("employees.parquet")
df_parquet.show()

# Write to CSV
df.write.csv("employees.csv", mode="overwrite", header=True)
print("Written to CSV")

# Read from CSV with options
df_csv = spark.read.csv("employees.csv", header=True, inferSchema=True)
df_csv.show()

# Write to JSON
df.write.json("employees.json", mode="overwrite")
print("Written to JSON")

# Read from JSON
df_json = spark.read.json("employees.json")
df_json.show()

## 9. Performance Tuning

- **Partitioning**: Control number of partitions
- **Caching / Persistence**: Keep intermediate results in memory
- **Broadcast joins**: Small table broadcasted to all nodes
- **Salting**: Avoid data skew
- **Column pruning**: Select only needed columns

In [ ]:
# Partitioning
print(f"Number of partitions: {df.rdd.getNumPartitions()}")
df_repartitioned = df.repartition(10)  # Increase/decrease partitions
df_coalesced = df.coalesce(1)  # Reduce partitions (no shuffle)

# Caching
df.cache()  # Cache in memory
# df.persist(StorageLevel.MEMORY_AND_DISK)
df.count()  # Materialize cache
print(f"Is cached: {df.is_cached}")
df.unpersist()  # Remove from cache

# Broadcast hint for small tables
small = spark.createDataFrame([(1, "Alice")], ["id", "name"])
large = spark.range(1000).repartition(100)

# Using broadcast hint
from pyspark.sql.functions import broadcast
joined = large.join(broadcast(small), large.id == small.id)
joined.explain()

## 10. Catalyst Optimizer and Tungsten

Spark uses Catalyst for query optimization and Tungsten for efficient memory management.

- **Explain plans**: `df.explain()` shows physical plan
- **Adaptive Query Execution (AQE)**: Dynamically optimizes at runtime

In [ ]:
# Show execution plan
df.filter(df.Age > 30).groupBy("Department").count().explain(mode="formatted")

# Show logical and physical plans
# df.explain(True)

## 11. User Defined Functions (UDFs)

UDFs allow you to define custom Python functions, but they are slower than built-in functions. Use Pandas UDFs (Vectorized UDFs) for better performance.

In [ ]:
# Simple UDF
from pyspark.sql.types import StringType

def age_category(age):
    if age < 30:
        return "Young"
    elif age < 40:
        return "Middle"
    else:
        return "Senior"

age_udf = F.udf(age_category, StringType())
df.withColumn("AgeCategory", age_udf(df.Age)).show()

# Pandas UDF (Vectorized, much faster)
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(StringType())
def age_category_pandas(age: pd.Series) -> pd.Series:
    return age.apply(lambda x: "Young" if x < 30 else ("Middle" if x < 40 else "Senior"))

df.withColumn("AgeCategory", age_category_pandas(df.Age)).show()

## 12. Machine Learning with MLlib

MLlib provides scalable machine learning algorithms (classification, regression, clustering, collaborative filtering).

In [ ]:
# Sample classification with logistic regression
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Load Iris-like data
iris_data = [(1.5, 0.4, "setosa"), (3.0, 1.2, "versicolor"), (3.5, 1.5, "virginica")]
iris_df = spark.createDataFrame(iris_data, ["petal_length", "petal_width", "species"])

# Index label column
indexer = StringIndexer(inputCol="species", outputCol="label")
indexed = indexer.fit(iris_df).transform(iris_df)

# Assemble features
assembler = VectorAssembler(inputCols=["petal_length", "petal_width"], outputCol="features")
data = assembler.transform(indexed)

# Split data
train, test = data.randomSplit([0.7, 0.3], seed=42)

# Train model
lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train)

# Predict
predictions = model.transform(test)
predictions.show()

# Evaluate
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy}")

# Clustering example (KMeans)
from pyspark.ml.clustering import KMeans
kmeans = KMeans(featuresCol="features", k=3)
kmeans_model = kmeans.fit(data)
print(f"Cluster centers: {kmeans_model.clusterCenters()}")

## 13. Structured Streaming

Process real-time data streams using the same DataFrame API.

In [ ]:
# Example: read stream from socket (run netcat in terminal: nc -lk 9999)
"""
# Read stream
lines = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

# Process
word_count = lines.select(explode(split(lines.value, " ")).alias("word")) \
                .groupBy("word") \
                .count()

# Write to console
query = word_count.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

# Stop stream
# query.stop()
"""
print("Structured Streaming enables real-time processing. Above is a socket streaming example.")

# Checkpointing and watermarks
# query = word_count.writeStream \
#     .outputMode("append") \
#     .format("parquet") \
#     .option("checkpointLocation", "/path/to/checkpoint") \
#     .start("/path/to/output")

## 14. Graph Processing (GraphX)

GraphX is for graph-parallel computation. In PySpark, you can use GraphFrames (external package).

In [ ]:
# GraphFrames requires separate installation: spark-submit --packages graphframes:graphframes:0.8.2-spark3.2-s_2.12
# Example with GraphFrames:
"""
from graphframes import GraphFrame

vertices = spark.createDataFrame([("a", "Alice"), ("b", "Bob"), ("c", "Charlie")], ["id", "name"])
edges = spark.createDataFrame([("a", "b", "friend"), ("b", "c", "follow")], ["src", "dst", "relationship"])

g = GraphFrame(vertices, edges)
g.vertices.show()
g.edges.show()

# PageRank
results = g.pageRank(resetProbability=0.15, maxIter=10)
results.vertices.select("id", "pagerank").show()
"""
print("GraphX/GraphFrames are available for graph algorithms (PageRank, shortest paths, connected components).")

## 15. Best Practices

1. **Use DataFrames over RDDs** – Catalyst optimizations make them faster.
2. **Avoid UDFs when possible** – use built‑in functions.
3. **Use broadcast joins** for small tables.
4. **Partition your data** appropriately (avoid too many small files).
5. **Cache intermediate results** that are reused.
6. **Use `select` instead of `selectExpr`** for clarity (both fine).
7. **Enable AQE** – `spark.sql.adaptive.enabled=true`.
8. **Tune shuffle partitions** – `spark.sql.shuffle.partitions`.
9. **Write to Parquet** instead of CSV for performance.
10. **Use `explain()`** to understand query plans.

In [ ]:
# Example configuration settings
print("Current shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Adaptive execution enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

# Set configurations (in builder or dynamically)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Memory and core tuning (in spark-submit or config)
# --executor-memory 4g
# --executor-cores 2
# --driver-memory 2g

## Summary

You have now covered:
- **RDD basics** – transformations, actions, key-value operations
- **DataFrames** – creation, selection, filtering, aggregations, sorting
- **Spark SQL** – temporary views, SQL queries
- **Advanced column operations** – withColumn, when, string/array functions
- **Window functions** – ranking, lag/lead, running totals
- **Joins** – all standard types
- **Missing data** – dropna, fillna
- **Data sources** – reading/writing Parquet, CSV, JSON
- **Performance tuning** – partitioning, caching, broadcast joins
- **Catalyst & Tungsten** – explain plans, AQE
- **UDFs** – Python UDFs and Pandas UDFs
- **Machine Learning with MLlib** – feature engineering, classification, clustering
- **Structured Streaming** – real‑time processing
- **Graph processing** – GraphX / GraphFrames
- **Best practices** – optimisation and deployment

PySpark is the go‑to framework for large‑scale data processing. Practice with real datasets to master these concepts.

In [ ]:
# Stop Spark session (good practice at the end)
# spark.stop()
print("Spark session can be stopped with spark.stop() when finished.")